# 03. Feature Engineering para el sistema Text-to-SQL — ONPE 2026

## Propósito

Preparar las estructuras analíticas derivadas de **todos los tipos de JSON considerados en el EDA**:

- `participantes.json`: resultados por organización política.
- `totales.json`: totales electorales y UBIGEO.
- `departamentos.json`: catálogo de departamentos.
- `provincias.json`: catálogo de provincias.
- `distritos.json`: catálogo de distritos.

## 1. Configuración y reproducibilidad

In [2]:
from pathlib import Path
import json
import hashlib
import logging
import platform
import sys
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Directorio raíz: si el notebook está dentro de /notebooks, sube un nivel.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Si la detección automática no coincide con tu estructura, descomenta y ajusta:
# PROJECT_ROOT = Path(r"C:/ruta/a/tu/proyecto")

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ONPE_2026" / "Peru"
OUTPUT_DIR = PROJECT_ROOT / "results" / "feature_engineering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 31
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (10, 5)

# Logging reproducible de la ejecución.
LOG_PATH = OUTPUT_DIR / "feature_engineering.log"
logger = logging.getLogger("onpe_feature_engineering")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(_file_handler)

print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos:", DATA_DIR)
print("Resultados:", OUTPUT_DIR)

Raíz del proyecto: c:\Users\Kencho\Desktop\MaestriaIA\Trabajo Investigacion II\Repositorio\ProyectoTesis
Datos: c:\Users\Kencho\Desktop\MaestriaIA\Trabajo Investigacion II\Repositorio\ProyectoTesis\data\raw\ONPE_2026\Peru
Resultados: c:\Users\Kencho\Desktop\MaestriaIA\Trabajo Investigacion II\Repositorio\ProyectoTesis\results\feature_engineering


## 2. Verificación e inventario de los JSON

Se localizan recursivamente los cinco tipos de archivo.

In [3]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"No existe el directorio de datos: {DATA_DIR}\n"
        "Verifica la estructura del proyecto o ajusta PROJECT_ROOT."
    )

json_files = sorted(DATA_DIR.rglob("*.json"))

EXPECTED_FILES = [
    "participantes.json",
    "totales.json",
    "departamentos.json",
    "provincias.json",
    "distritos.json",
]

inventory = pd.DataFrame([
    {
        "archivo": name,
        "cantidad": sum(p.name.lower() == name for p in json_files)
    }
    for name in EXPECTED_FILES
])

display(inventory)

missing_types = inventory.loc[inventory["cantidad"].eq(0), "archivo"].tolist()
if missing_types:
    raise FileNotFoundError(f"No se encontraron estos tipos de JSON: {missing_types}")

logger.info("JSON encontrados: %s", len(json_files))
logger.info("Inventario por tipo: %s", inventory.to_dict(orient="records"))

,archivo,cantidad
0,participantes.json,2113
1,totales.json,2113
2,departamentos.json,1
3,provincias.json,25
4,distritos.json,196


## 3. Funciones de lectura y normalización

Los catálogos y resultados utilizan una envoltura JSON con la propiedad `data`. La función siguiente contempla tanto `data` como estructuras de lista u objeto, y conserva la ruta relativa del archivo fuente.

Los UBIGEO se almacenan como cadenas para preservar ceros a la izquierda y evitar tratarlos como cantidades numéricas.

In [4]:
def unwrap_json_data(payload):
    """Devuelve el contenido de data cuando existe; en otro caso devuelve el payload."""
    if isinstance(payload, dict) and "data" in payload:
        return payload["data"]
    return payload


def records_from_json(payload):
    """Convierte listas u objetos JSON en registros tabulares."""
    data = unwrap_json_data(payload)

    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]

    if isinstance(data, dict):
        return [data]

    return []


def load_json_type(filename):
    """Carga todos los archivos de un tipo y conserva source_file/source_dir."""
    frames, errors = [], []

    for path in json_files:
        if path.name.lower() != filename.lower():
            continue

        try:
            with path.open("r", encoding="utf-8") as f:
                payload = json.load(f)

            records = records_from_json(payload)
            if not records:
                errors.append({
                    "source_file": str(path.relative_to(DATA_DIR)),
                    "error": "No se encontraron registros tipo objeto en data."
                })
                continue

            frame = pd.json_normalize(records, sep=".")
            relative_path = path.relative_to(DATA_DIR)
            frame["source_file"] = str(relative_path)
            frame["source_dir"] = str(relative_path.parent)
            frames.append(frame)

        except Exception as exc:
            errors.append({
                "source_file": str(path.relative_to(DATA_DIR)),
                "error": repr(exc)
            })

    result = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    return result, pd.DataFrame(errors)


def normalize_ubigeo(series, expected_length=None):
    """Normaliza UBIGEO como texto; no convierte códigos a números."""
    values = series.astype("string").str.strip()
    values = values.str.replace(r"\.0$", "", regex=True)
    if expected_length:
        values = values.str.zfill(expected_length)
    return values


def to_numeric_columns(df, columns):
    """Convierte columnas existentes a numéricas sin abortar por valores inválidos."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def normalize_text(value):
    """Normaliza texto para comparaciones; conserva aparte el nombre original."""
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", value).strip().upper()

## 4. Carga de los cinco tipos de JSON

Se cargan los resultados por organización, los totales por distrito y los tres catálogos geográficos. No se utiliza `dniCandidato` en las tablas analíticas, porque no es necesario para las preguntas de investigación planteadas.

In [5]:
df_participantes_raw, errors_participantes = load_json_type("participantes.json")
df_totales_raw, errors_totales = load_json_type("totales.json")
df_departamentos_raw, errors_departamentos = load_json_type("departamentos.json")
df_provincias_raw, errors_provincias = load_json_type("provincias.json")
df_distritos_raw, errors_distritos = load_json_type("distritos.json")

loaded = {
    "participantes": len(df_participantes_raw),
    "totales": len(df_totales_raw),
    "departamentos": len(df_departamentos_raw),
    "provincias": len(df_provincias_raw),
    "distritos": len(df_distritos_raw),
}
display(pd.DataFrame([{"tipo_json": k, "registros": v} for k, v in loaded.items()]))

errors = {
    "participantes": errors_participantes,
    "totales": errors_totales,
    "departamentos": errors_departamentos,
    "provincias": errors_provincias,
    "distritos": errors_distritos,
}
for name, error_df in errors.items():
    print(f"{name}: {len(error_df)} errores")
    if not error_df.empty:
        display(error_df.head())

logger.info("Registros cargados: %s", loaded)

,tipo_json,registros
0,participantes,76068
1,totales,2113
2,departamentos,25
3,provincias,196
4,distritos,1892


participantes: 0 errores
totales: 0 errores
departamentos: 0 errores
provincias: 0 errores
distritos: 0 errores


## 5. Inspección de esquema y selección de columnas

Antes de transformar, se visualizan las columnas reales encontradas. El notebook valida los campos que necesita para construir las tablas canónicas; si la estructura de origen cambia, se detiene con un mensaje que indica qué campo falta.

In [ ]:
for name, frame in [
    ("participantes", df_participantes_raw),
    ("totales", df_totales_raw),
    ("departamentos", df_departamentos_raw),
    ("provincias", df_provincias_raw),
    ("distritos", df_distritos_raw),
]:
    print(f"\\n===== {name.upper()} =====")
    print("Dimensiones:", frame.shape)
    print("Columnas:", frame.columns.tolist())
    display(frame.head(2))

In [ ]:
# Los campos de participantes corresponden a la estructura observada en los JSON del EDA.
PARTY_COL = "nombreAgrupacionPolitica"
PARTY_CODE_COL = "codigoAgrupacionPolitica"
CANDIDATE_COL = "nombreCandidato"
VOTES_COL = "totalVotosValidos"

# Los campos de totales están anidados bajo data y quedan aplanados con el prefijo data.
TOTAL_DISTRICT_COL = "idUbigeoDistrito"
TOTAL_PROVINCE_COL = "idUbigeoProvincia"
TOTAL_DEPARTMENT_COL = "idUbigeoDepartamento"
TOTAL_VALID_COL = "totalVotosValidos"
TOTAL_EMITTED_COL = "totalVotosEmitidos"
TOTAL_ACTAS_COL = "totalActas"
COUNTED_ACTAS_COL = "contabilizadas"
ACTAS_PERCENT_COL = "actasContabilizadas"
TURNOUT_PERCENT_COL = "participacionCiudadana"

def require_columns(df, columns, dataset_name):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(
            f"En {dataset_name} faltan columnas requeridas: {missing}. "
            f"Columnas disponibles: {df.columns.tolist()}"
        )

require_columns(
    df_participantes_raw,
    [PARTY_COL, PARTY_CODE_COL, VOTES_COL, "source_dir", "source_file"],
    "participantes.json"
)
require_columns(
    df_totales_raw,
    [TOTAL_DISTRICT_COL, TOTAL_PROVINCE_COL, TOTAL_DEPARTMENT_COL,
     TOTAL_VALID_COL, "source_dir", "source_file"],
    "totales.json"
)
for catalog_name, catalog in [
    ("departamentos.json", df_departamentos_raw),
    ("provincias.json", df_provincias_raw),
    ("distritos.json", df_distritos_raw),
]:
    require_columns(catalog, ["ubigeo", "nombre"], catalog_name)

print("Validación de columnas requeridas: correcta.")

## 6. Construcción de dimensiones geográficas

Se construyen tres dimensiones con una fila por UBIGEO. Los catálogos de provincia y distrito aparecen repetidos en carpetas de su nivel territorial; por eso se eliminan duplicados por código y se comprueba que un mismo código no tenga nombres contradictorios.

Jerarquía empleada:
- Departamento: UBIGEO de 6 caracteres terminado en `0000`.
- Provincia: UBIGEO de 6 caracteres terminado en `00`.
- Distrito: UBIGEO de 6 caracteres.

In [ ]:
def build_geo_dimension(df, level_name, code_col="ubigeo", name_col="nombre"):
    dim = df[[code_col, name_col]].copy()
    dim = dim.rename(columns={code_col: "ubigeo", name_col: f"{level_name}_nombre"})
    dim["ubigeo"] = normalize_ubigeo(dim["ubigeo"], 6)
    dim[f"{level_name}_nombre"] = dim[f"{level_name}_nombre"].astype("string").str.strip()
    dim = dim.dropna(subset=["ubigeo"]).drop_duplicates()

    conflicts = (
        dim.groupby("ubigeo")[f"{level_name}_nombre"]
        .nunique(dropna=True)
        .loc[lambda x: x > 1]
    )
    if not conflicts.empty:
        raise ValueError(
            f"Hay UBIGEO con nombres contradictorios en {level_name}: "
            f"{conflicts.index.tolist()[:10]}"
        )

    return dim.drop_duplicates("ubigeo").reset_index(drop=True)


dim_departamento = build_geo_dimension(df_departamentos_raw, "departamento")
dim_provincia = build_geo_dimension(df_provincias_raw, "provincia")
dim_distrito = build_geo_dimension(df_distritos_raw, "distrito")

dim_departamento["ubigeo_departamento"] = dim_departamento["ubigeo"]
dim_provincia["ubigeo_provincia"] = dim_provincia["ubigeo"]
dim_provincia["ubigeo_departamento"] = dim_provincia["ubigeo"].str[:2] + "0000"
dim_distrito["ubigeo_distrito"] = dim_distrito["ubigeo"]
dim_distrito["ubigeo_provincia"] = dim_distrito["ubigeo"].str[:4] + "00"
dim_distrito["ubigeo_departamento"] = dim_distrito["ubigeo"].str[:2] + "0000"

# Tabla geográfica consolidada: una fila por distrito.
dim_geografia = (
    dim_distrito[["ubigeo_distrito", "distrito_nombre",
                  "ubigeo_provincia", "ubigeo_departamento"]]
    .merge(
        dim_provincia[["ubigeo_provincia", "provincia_nombre"]],
        on="ubigeo_provincia", how="left", validate="many_to_one"
    )
    .merge(
        dim_departamento[["ubigeo_departamento", "departamento_nombre"]],
        on="ubigeo_departamento", how="left", validate="many_to_one"
    )
)

print("Departamentos:", len(dim_departamento))
print("Provincias:", len(dim_provincia))
print("Distritos:", len(dim_distrito))
print("Filas en dimensión geográfica:", len(dim_geografia))
display(dim_geografia.head())

## 7. Construcción de la tabla de resultados por organización y distrito

La tabla de hechos `fact_votos_distrito` tendrá una fila por distrito y organización política. Se relaciona `participantes.json` con `totales.json` usando la carpeta de origen (`source_dir`), ya que ambos archivos corresponden al mismo distrito.

Se preservan los nombres originales de organización y candidato. No se conserva DNI. Los valores de votos y porcentajes se convierten a tipo numérico.

In [ ]:
# Preparar participantes: una fila por organización política en cada carpeta distrital.
fact_votos_distrito = df_participantes_raw[
    ["source_dir", "source_file", PARTY_CODE_COL, PARTY_COL, CANDIDATE_COL, VOTES_COL]
].copy()

fact_votos_distrito = fact_votos_distrito.rename(columns={
    PARTY_CODE_COL: "codigo_organizacion",
    PARTY_COL: "organizacion_politica",
    CANDIDATE_COL: "candidato",
    VOTES_COL: "votos_validos"
})

fact_votos_distrito["votos_validos"] = pd.to_numeric(
    fact_votos_distrito["votos_validos"], errors="coerce"
)
fact_votos_distrito["codigo_organizacion"] = pd.to_numeric(
    fact_votos_distrito["codigo_organizacion"], errors="coerce"
).astype("Int64")

# Clave del distrito a partir de totales.json.
totales_key = df_totales_raw[
    ["source_dir", TOTAL_DISTRICT_COL, TOTAL_PROVINCE_COL, TOTAL_DEPARTMENT_COL,
     TOTAL_VALID_COL, TOTAL_EMITTED_COL, "source_file"]
].copy()

totales_key = totales_key.rename(columns={
    TOTAL_DISTRICT_COL: "ubigeo_distrito_fuente",
    TOTAL_PROVINCE_COL: "ubigeo_provincia_fuente",
    TOTAL_DEPARTMENT_COL: "ubigeo_departamento_fuente",
    TOTAL_VALID_COL: "total_votos_validos_distrito",
    TOTAL_EMITTED_COL: "total_votos_emitidos"
})

# El árbol de carpetas también contiene los códigos UBIGEO (ej. 010000/010100/010101).
# Se derivan las claves territoriales desde la ruta y se conservan los códigos del JSON
# por separado para poder detectar inconsistencias sin asignar geografía incorrecta.
def ubigeos_from_source_dir(source_dir):
    parts = Path(source_dir).parts
    codes = []
    for part in parts:
        match = re.match(r"^(\\d{6})_", part)
        if match:
            codes.append(match.group(1))
    if len(codes) < 3:
        return pd.Series({
            "ubigeo_departamento": pd.NA,
            "ubigeo_provincia": pd.NA,
            "ubigeo_distrito": pd.NA
        })
    return pd.Series({
        "ubigeo_departamento": codes[-3],
        "ubigeo_provincia": codes[-2],
        "ubigeo_distrito": codes[-1]
    })

path_codes = totales_key["source_dir"].apply(ubigeos_from_source_dir)
totales_key = pd.concat([totales_key.reset_index(drop=True), path_codes.reset_index(drop=True)], axis=1)

for col in ["ubigeo_distrito_fuente", "ubigeo_provincia_fuente", "ubigeo_departamento_fuente"]:
    totales_key[col] = normalize_ubigeo(totales_key[col], 6)

totales_key["ubigeo_distrito_fuente_coincide_ruta"] = (
    totales_key["ubigeo_distrito_fuente"] == totales_key["ubigeo_distrito"]
)
mismatch_source_ubigeo = totales_key.loc[
    ~totales_key["ubigeo_distrito_fuente_coincide_ruta"],
    ["source_dir", "ubigeo_distrito_fuente", "ubigeo_distrito"]
]
print("Registros cuyo UBIGEO de totales.json no coincide con la carpeta:", len(mismatch_source_ubigeo))
if not mismatch_source_ubigeo.empty:
    display(mismatch_source_ubigeo.head(20))

for col in ["total_votos_validos_distrito", "total_votos_emitidos"]:
    totales_key[col] = pd.to_numeric(totales_key[col], errors="coerce")

# Cada source_dir debe identificar un único registro de totales por distrito.
duplicated_total_dirs = totales_key["source_dir"].duplicated(keep=False)
if duplicated_total_dirs.any():
    display(totales_key.loc[duplicated_total_dirs].sort_values("source_dir").head(20))
    raise ValueError("Hay más de un registro totales.json asociado a una misma carpeta.")

fact_votos_distrito = fact_votos_distrito.merge(
    totales_key[[
        "source_dir", "ubigeo_distrito", "ubigeo_provincia",
        "ubigeo_departamento", "total_votos_validos_distrito",
        "total_votos_emitidos"
    ]],
    on="source_dir", how="left", validate="many_to_one"
)

fact_votos_distrito = fact_votos_distrito.merge(
    dim_geografia,
    on=["ubigeo_distrito", "ubigeo_provincia", "ubigeo_departamento"],
    how="left",
    validate="many_to_one"
)

# Participación de cada organización dentro de los votos válidos del distrito.
fact_votos_distrito["porcentaje_votos_validos"] = np.where(
    fact_votos_distrito["total_votos_validos_distrito"].gt(0),
    fact_votos_distrito["votos_validos"] /
    fact_votos_distrito["total_votos_validos_distrito"] * 100,
    np.nan
)

print("Filas fact_votos_distrito:", len(fact_votos_distrito))
display(fact_votos_distrito.head())

## 8. Construcción de la tabla de totales por distrito

Esta tabla conserva las métricas del archivo `totales.json` que son útiles para consultas sobre participación, actas y votos. Los campos opcionales se incorporan solo si están presentes en los archivos.

In [ ]:
optional_total_map = {
    "totalVotosBlancos": "votos_blancos",
    "totalVotosNulos": "votos_nulos",
    "totalVotosImpugnados": "votos_impugnados",
    "totalElectoresHabiles": "electores_habiles",
    "actasContabilizadas": "porcentaje_actas_contabilizadas",
    "contabilizadas": "actas_contabilizadas",
    "totalActas": "total_actas",
    "participacionCiudadana": "porcentaje_participacion_ciudadana",
    "porcentajeVotosValidos": "porcentaje_votos_validos_fuente",
    "porcentajeVotosEmitidos": "porcentaje_votos_emitidos_fuente",
    "fechaActualizacion": "fecha_actualizacion_epoch_ms",
}

base_total_cols = [
    "source_dir", "ubigeo_distrito", "ubigeo_provincia", "ubigeo_departamento",
    "total_votos_validos_distrito", "total_votos_emitidos",
    "ubigeo_distrito_fuente", "ubigeo_provincia_fuente", "ubigeo_departamento_fuente",
    "ubigeo_distrito_fuente_coincide_ruta"
]
for source_col, target_col in optional_total_map.items():
    if source_col in df_totales_raw.columns:
        totales_key[target_col] = df_totales_raw[source_col].values
        base_total_cols.append(target_col)

fact_totales_distrito = totales_key[base_total_cols].copy()

numeric_fact_cols = [
    c for c in fact_totales_distrito.columns
    if c not in ["source_dir", "ubigeo_distrito", "ubigeo_provincia", "ubigeo_departamento",
                  "ubigeo_distrito_fuente", "ubigeo_provincia_fuente", "ubigeo_departamento_fuente"]
]
fact_totales_distrito = to_numeric_columns(fact_totales_distrito, numeric_fact_cols)

fact_totales_distrito = fact_totales_distrito.merge(
    dim_geografia,
    on=["ubigeo_distrito", "ubigeo_provincia", "ubigeo_departamento"],
    how="left",
    validate="many_to_one"
)

display(fact_totales_distrito.head())

## 9. Feature Engineering electoral: ranking, ganador y margen

Se generan variables derivadas por distrito:

- `ranking_votos`: posición de la organización según votos válidos.
- `es_ganador_distrital`: identifica la organización con mayor cantidad de votos.
- `votos_segundo_lugar`: votos de la segunda organización.
- `margen_votos_primero_segundo`: diferencia de votos entre primero y segundo.
- `margen_porcentual_primero_segundo`: margen como porcentaje de votos válidos.

En caso de empate en el primer lugar, todas las organizaciones empatadas quedan marcadas como ganadoras y el margen es cero.

In [ ]:
# Ranking denso: las organizaciones empatadas comparten posición.
fact_votos_distrito["ranking_votos"] = (
    fact_votos_distrito.groupby("source_dir")["votos_validos"]
    .rank(method="dense", ascending=False)
)

max_votes = fact_votos_distrito.groupby("source_dir")["votos_validos"].transform("max")
fact_votos_distrito["es_ganador_distrital"] = (
    fact_votos_distrito["votos_validos"].eq(max_votes) & max_votes.notna()
)

# Obtener primero y segundo puesto por distrito.
ranked = fact_votos_distrito.sort_values(
    ["source_dir", "votos_validos", "codigo_organizacion"],
    ascending=[True, False, True]
).copy()

district_top2 = (
    ranked.groupby("source_dir")
    .agg(
        votos_primer_lugar=("votos_validos", "max"),
        votos_segundo_lugar=("votos_validos",
                             lambda s: s.sort_values(ascending=False).iloc[1]
                             if len(s) > 1 else np.nan)
    )
    .reset_index()
)

district_top2["margen_votos_primero_segundo"] = (
    district_top2["votos_primer_lugar"] - district_top2["votos_segundo_lugar"]
)
district_top2 = district_top2.merge(
    fact_totales_distrito[["source_dir", "total_votos_validos_distrito"]],
    on="source_dir", how="left", validate="one_to_one"
)
district_top2["margen_porcentual_primero_segundo"] = np.where(
    district_top2["total_votos_validos_distrito"].gt(0),
    district_top2["margen_votos_primero_segundo"] /
    district_top2["total_votos_validos_distrito"] * 100,
    np.nan
)

fact_votos_distrito = fact_votos_distrito.merge(
    district_top2[[
        "source_dir", "votos_primer_lugar", "votos_segundo_lugar",
        "margen_votos_primero_segundo", "margen_porcentual_primero_segundo"
    ]],
    on="source_dir", how="left", validate="many_to_one"
)

display(fact_votos_distrito[
    ["departamento_nombre", "provincia_nombre", "distrito_nombre",
     "organizacion_politica", "votos_validos", "ranking_votos",
     "es_ganador_distrital", "margen_votos_primero_segundo"]
].head(15))

## 10. Feature Engineering electoral: concentración y distribución de votos

Se calculan indicadores por distrito a partir de la proporción de votos de cada organización:

- `cuota_votos`: participación proporcional de cada organización.
- `hhi_concentracion_votos`: suma de los cuadrados de las cuotas de voto (escala 0–1).
- `numero_efectivo_organizaciones`: inverso del HHI cuando es positivo.
- `cuota_votos_top2`: suma de las cuotas de las dos organizaciones más votadas.

El HHI se presenta como indicador descriptivo de concentración electoral, no como una medida de calidad democrática.

In [ ]:
fact_votos_distrito["cuota_votos"] = np.where(
    fact_votos_distrito["total_votos_validos_distrito"].gt(0),
    fact_votos_distrito["votos_validos"] /
    fact_votos_distrito["total_votos_validos_distrito"],
    np.nan
)

district_concentration = (
    fact_votos_distrito.groupby("source_dir")
    .agg(
        hhi_concentracion_votos=("cuota_votos", lambda s: np.square(s.dropna()).sum()
                                 if s.notna().any() else np.nan),
        cuota_votos_top2=("cuota_votos", lambda s: s.nlargest(2).sum()
                          if s.notna().any() else np.nan),
        organizaciones_con_votos=("votos_validos", lambda s: int((s > 0).sum()))
    )
    .reset_index()
)

district_concentration["numero_efectivo_organizaciones"] = np.where(
    district_concentration["hhi_concentracion_votos"].gt(0),
    1 / district_concentration["hhi_concentracion_votos"],
    np.nan
)

fact_totales_distrito = fact_totales_distrito.merge(
    district_concentration,
    on="source_dir", how="left", validate="one_to_one"
)

display(district_concentration.head())

## 11. Atributos de participación y actas

Se calculan únicamente cuando los campos de origen están disponibles:

- `porcentaje_votos_validos_calculado`: votos válidos / votos emitidos × 100.
- `porcentaje_votos_no_validos_calculado`: (votos emitidos − votos válidos) / votos emitidos × 100.

No se infiere participación electoral a partir de votos emitidos si el JSON no proporciona el total de electores hábiles. El porcentaje `participacionCiudadana` del archivo de origen se conserva con su nombre explícito.

In [ ]:
fact_totales_distrito["porcentaje_votos_validos_calculado"] = np.where(
    fact_totales_distrito["total_votos_emitidos"].gt(0),
    fact_totales_distrito["total_votos_validos_distrito"] /
    fact_totales_distrito["total_votos_emitidos"] * 100,
    np.nan
)

fact_totales_distrito["porcentaje_votos_no_validos_calculado"] = np.where(
    fact_totales_distrito["total_votos_emitidos"].gt(0),
    (
        fact_totales_distrito["total_votos_emitidos"]
        - fact_totales_distrito["total_votos_validos_distrito"]
    ) / fact_totales_distrito["total_votos_emitidos"] * 100,
    np.nan
)

display(fact_totales_distrito.head())

## 12. Agregaciones derivadas por provincia y departamento

Se generan tablas agregadas que permiten consultas frecuentes del sistema Text-to-SQL. Los votos se suman desde la tabla de hechos a nivel distrito–organización; los ganadores se calculan después de agregar los votos al nivel territorial correspondiente.

Esto evita asumir que el partido que gana más distritos necesariamente obtiene más votos totales.

In [ ]:
def aggregate_votes_by_level(fact, geo_key, geo_name):
    group_cols = [geo_key, geo_name, "codigo_organizacion", "organizacion_politica"]
    result = (
        fact.groupby(group_cols, dropna=False)
        .agg(votos_validos=("votos_validos", "sum"))
        .reset_index()
    )
    result["ranking_votos_nivel"] = (
        result.groupby(geo_key)["votos_validos"]
        .rank(method="dense", ascending=False)
    )
    max_by_level = result.groupby(geo_key)["votos_validos"].transform("max")
    result["es_ganador_nivel"] = result["votos_validos"].eq(max_by_level)
    return result


fact_votos_provincia = aggregate_votes_by_level(
    fact_votos_distrito, "ubigeo_provincia", "provincia_nombre"
)
fact_votos_departamento = aggregate_votes_by_level(
    fact_votos_distrito, "ubigeo_departamento", "departamento_nombre"
)

# Conteo de distritos ganados por organización dentro de cada departamento.
district_wins_by_department = (
    fact_votos_distrito.loc[fact_votos_distrito["es_ganador_distrital"]]
    .groupby(["ubigeo_departamento", "departamento_nombre",
              "codigo_organizacion", "organizacion_politica"])
    .agg(distritos_ganados=("ubigeo_distrito", "nunique"))
    .reset_index()
    .sort_values(["ubigeo_departamento", "distritos_ganados"],
                 ascending=[True, False])
)

display(fact_votos_provincia.head())
display(fact_votos_departamento.head())
display(district_wins_by_department.head())

## 13. Validaciones de integridad y consistencia

Las validaciones comprueban claves únicas en dimensiones, relaciones geográficas, ausencia de claves sin correspondencia y conciliación entre suma de votos por organización y total de votos válidos del distrito.

La conciliación es una comprobación esencial: si hay diferencias, deben analizarse antes de utilizar las tablas como base del sistema Text-to-SQL.

In [ ]:
validation_rows = []

def add_validation(name, condition, detail):
    validation_rows.append({
        "validacion": name,
        "resultado": "OK" if bool(condition) else "REVISAR",
        "detalle": detail
    })

add_validation(
    "UBIGEO único en dim_departamento",
    dim_departamento["ubigeo_departamento"].is_unique,
    f"Filas={len(dim_departamento)}, únicos={dim_departamento['ubigeo_departamento'].nunique()}"
)
add_validation(
    "UBIGEO único en dim_provincia",
    dim_provincia["ubigeo_provincia"].is_unique,
    f"Filas={len(dim_provincia)}, únicos={dim_provincia['ubigeo_provincia'].nunique()}"
)
add_validation(
    "UBIGEO único en dim_distrito",
    dim_distrito["ubigeo_distrito"].is_unique,
    f"Filas={len(dim_distrito)}, únicos={dim_distrito['ubigeo_distrito'].nunique()}"
)

missing_geo = int(fact_votos_distrito["departamento_nombre"].isna().sum())
add_validation(
    "Resultados vinculados a catálogo geográfico",
    missing_geo == 0,
    f"Filas sin departamento asociado={missing_geo}"
)

sum_votes = (
    fact_votos_distrito.groupby("source_dir")["votos_validos"]
    .sum(min_count=1)
    .rename("suma_votos_organizaciones")
    .reset_index()
)
reconciliation = sum_votes.merge(
    fact_totales_distrito[["source_dir", "ubigeo_distrito", "total_votos_validos_distrito"]],
    on="source_dir", how="outer", validate="one_to_one"
)
reconciliation["diferencia_votos"] = (
    reconciliation["suma_votos_organizaciones"]
    - reconciliation["total_votos_validos_distrito"]
)
reconciliation["coincide"] = np.isclose(
    reconciliation["suma_votos_organizaciones"],
    reconciliation["total_votos_validos_distrito"],
    equal_nan=False
)

add_validation(
    "Conciliación de votos por distrito",
    reconciliation["coincide"].all(),
    f"Distritos conciliados={int(reconciliation['coincide'].sum())}; "
    f"con diferencias={int((~reconciliation['coincide']).sum())}"
)

add_validation(
    "Votos no negativos",
    fact_votos_distrito["votos_validos"].dropna().ge(0).all(),
    "Se revisa que los votos por organización no sean negativos."
)
add_validation(
    "Cuotas de voto dentro del rango [0, 1]",
    fact_votos_distrito["cuota_votos"].dropna().between(0, 1).all(),
    "La cuota se calcula como votos de organización / votos válidos del distrito."
)

df_validations = pd.DataFrame(validation_rows)
display(df_validations)
display(reconciliation.loc[~reconciliation["coincide"]].head(20))

## 14. Diagnóstico visual de atributos derivados

Los gráficos sirven para inspeccionar distribuciones y detectar valores inesperados. No constituyen por sí solos evidencia de causalidad ni sustituyen las validaciones de integridad.

In [ ]:
# Distribución del margen porcentual entre primero y segundo.
margin_values = district_top2["margen_porcentual_primero_segundo"].dropna()
if not margin_values.empty:
    plt.figure(figsize=(9, 5))
    plt.hist(margin_values, bins=30)
    plt.title("Distribución del margen porcentual entre primero y segundo")
    plt.xlabel("Margen (% de votos válidos)")
    plt.ylabel("Cantidad de distritos")
    plt.grid(axis="y", alpha=0.25)
    plt.show()

# Distribución del HHI de concentración electoral.
hhi_values = fact_totales_distrito["hhi_concentracion_votos"].dropna()
if not hhi_values.empty:
    plt.figure(figsize=(9, 5))
    plt.hist(hhi_values, bins=30)
    plt.title("Distribución del HHI de concentración de votos por distrito")
    plt.xlabel("HHI (0–1)")
    plt.ylabel("Cantidad de distritos")
    plt.grid(axis="y", alpha=0.25)
    plt.show()

# Top departamentos por cantidad de distritos representados en la tabla.
district_count = (
    dim_geografia.groupby("departamento_nombre")["ubigeo_distrito"]
    .nunique().sort_values(ascending=False).head(15)
)
plt.figure(figsize=(10, 6))
district_count.sort_values().plot(kind="barh")
plt.title("Departamentos con mayor número de distritos en el catálogo")
plt.xlabel("Cantidad de distritos")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()

## 15. Diccionario de datos de salida

Se documentan las tablas que se utilizarán como insumo para diseñar el esquema relacional del sistema Text-to-SQL.

In [ ]:
data_dictionary = pd.DataFrame([
    ["dim_departamento", "Una fila por departamento", "ubigeo_departamento"],
    ["dim_provincia", "Una fila por provincia", "ubigeo_provincia"],
    ["dim_distrito", "Una fila por distrito", "ubigeo_distrito"],
    ["dim_geografia", "Jerarquía distrito–provincia–departamento", "ubigeo_distrito"],
    ["fact_votos_distrito", "Una fila por distrito y organización política", "ubigeo_distrito + codigo_organizacion"],
    ["fact_totales_distrito", "Totales y atributos derivados por distrito", "ubigeo_distrito"],
    ["fact_votos_provincia", "Votos agregados por provincia y organización", "ubigeo_provincia + codigo_organizacion"],
    ["fact_votos_departamento", "Votos agregados por departamento y organización", "ubigeo_departamento + codigo_organizacion"],
    ["district_wins_by_department", "Cantidad de distritos ganados por organización en cada departamento", "ubigeo_departamento + codigo_organizacion"],
], columns=["tabla", "granularidad", "clave_logica"])
display(data_dictionary)

## 16. Exportación de tablas y artefactos

Se exportan las tablas en CSV UTF-8 y Parquet cuando el motor local tiene instalado `pyarrow` o `fastparquet`. Los archivos son resultados reproducibles de esta etapa; no reemplazan los JSON originales.

In [ ]:
tables_to_export = {
    "dim_departamento": dim_departamento,
    "dim_provincia": dim_provincia,
    "dim_distrito": dim_distrito,
    "dim_geografia": dim_geografia,
    "fact_votos_distrito": fact_votos_distrito,
    "fact_totales_distrito": fact_totales_distrito,
    "fact_votos_provincia": fact_votos_provincia,
    "fact_votos_departamento": fact_votos_departamento,
    "district_wins_by_department": district_wins_by_department,
    "reconciliation_votos": reconciliation,
    "validaciones": df_validations,
    "data_dictionary": data_dictionary,
}

export_rows = []
for table_name, table in tables_to_export.items():
    csv_path = OUTPUT_DIR / f"{table_name}.csv"
    table.to_csv(csv_path, index=False, encoding="utf-8-sig")

    parquet_status = "no disponible"
    try:
        parquet_path = OUTPUT_DIR / f"{table_name}.parquet"
        table.to_parquet(parquet_path, index=False)
        parquet_status = "exportado"
    except (ImportError, ModuleNotFoundError):
        parquet_status = "requiere pyarrow o fastparquet"
    except Exception as exc:
        parquet_status = f"error: {exc}"

    export_rows.append({
        "tabla": table_name,
        "filas": len(table),
        "csv": str(csv_path),
        "parquet": parquet_status
    })

df_exports = pd.DataFrame(export_rows)
display(df_exports)

## 17. Metadatos y bitácora de ejecución

Se guarda la versión del entorno, la fecha de ejecución, la huella SHA-256 del conjunto de JSON y el resultado de las validaciones. Esto permite identificar exactamente qué datos generaron las tablas.

In [ ]:
dataset_hash = hashlib.sha256()
for path in json_files:
    dataset_hash.update(str(path.relative_to(DATA_DIR)).encode("utf-8"))
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            dataset_hash.update(chunk)

metadata = {
    "fecha_ejecucion": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version,
    "plataforma": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "random_seed": RANDOM_SEED,
    "data_dir": str(DATA_DIR),
    "cantidad_archivos_json": len(json_files),
    "inventario": loaded,
    "sha256_conjunto_json": dataset_hash.hexdigest(),
    "filas_fact_votos_distrito": len(fact_votos_distrito),
    "filas_fact_totales_distrito": len(fact_totales_distrito),
    "validaciones": df_validations.to_dict(orient="records"),
}

with (OUTPUT_DIR / "feature_engineering_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

logger.info("Feature Engineering finalizado.")
logger.info("SHA-256: %s", metadata["sha256_conjunto_json"])
logger.info("Validaciones: %s", df_validations.to_dict(orient="records"))

print("Proceso finalizado.")
print("Resultados guardados en:", OUTPUT_DIR)
print("Log:", LOG_PATH)